# NB09 — Ensembles, calibration, TTA, and conformal uncertainty

This notebook downloads only the selected Stage-A predictions/checkpoints from
the public HF dataset. It implements the four operations it claims: seed and
architecture ensembles, horizontal-flip TTA, temperature scaling on a disjoint
split, and conformal prediction sets on a second disjoint split.


In [ ]:
# === CELL 1 of every notebook: unpack the library ==========================
# Writes tyrelib.py into the session and imports it. Nothing here touches the
# GPU or the network beyond installing three small packages.
#
#   tyrelib   the whole pipeline: HuggingFace sync, registry, work sharding,
#             telemetry, model zoo, training loop, metrics.
#
# Generated by build_notebooks.py from tyrelib.py. Editing the blob below does
# nothing useful -- the next rebuild overwrites it. Edit the source instead.
import base64, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Kaggle ships torch, pandas, sklearn. These vary by image version, so check.
#   pynvml  reads GPU power/temperature/clocks directly (per device)
#   psutil  peak RAM and CPU
#   pyarrow writes per-sample predictions as Parquet
for _pkg in ('pynvml', 'psutil', 'pyarrow', 'timm'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'[BOOT] installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg],
                       check=False)

_LIB = (
    '...',
    'ZXBzKSkK',
)

(WORK / 'tyrelib.py').write_bytes(base64.b64decode(''.join(_LIB)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
# Without this, re-running cell 1 after an edit returns the cached module and
# you spend an hour debugging a ghost.
for _m in [m for m in list(sys.modules) if m == 'tyrelib']:
    del sys.modules[_m]
import tyrelib as tl
print('tyrelib', tl.__version__, 'loaded')


## 1 — Session, data, and locked architectures

In [ ]:
# === Who am I? =============================================================
#
# ACCOUNT   labels this Kaggle account in the shared run log. Two accounts
#           calling themselves the same thing makes the log useless.
# NUM_WORKERS  how many Kaggle accounts are running this notebook in parallel.
# WORKER_ID    0 .. NUM_WORKERS-1, DIFFERENT on each account.
#
# ---------------------------------------------------------------------------
# THESE TWO VALUES ARE SAFE TO CHANGE AT ANY TIME.
#
# They decide which account owns each fresh run. An absent or partial run stays
# with that static owner. Work stealing is disabled by default; an explicit
# recovery run may opt in and can then take over only a real claim/run event
# older than 45 minutes. Whether a run is finished, and what epoch it reached,
# is read from HuggingFace -- from the run's own files -- so it is the same
# answer for every account at every worker count.
# Go from 4 workers to 1 and nothing is retrained: the runs the other three
# finished are skipped, and the ones they left half-done are RESUMED from
# their checkpoints.
#
# (It did not always work that way. Resume used to check only the local disk,
#  and Kaggle wipes that between sessions, so every run restarted at epoch 1.
#  See docs/05 -- Bug 8.)
# ---------------------------------------------------------------------------
#
# All accounts push to the SAME HuggingFace account (Shanmuk4622), so the
# 128-writes-per-hour budget is SHARED. tyrelib caps each worker at
# 100/NUM_WORKERS automatically.
ACCOUNT     = 'acct1'   # <<< acct1 / acct2 / acct3 / acct4 on the four copies
NUM_WORKERS = 4         # <<< set 1 only when you are really running one notebook

# Derive the worker id from the account label so changing ACCOUNT is enough.
# This prevents four copies that all silently identify themselves as worker 0.
_ACCOUNT_TO_WORKER = {'acct1': 0, 'acct2': 1, 'acct3': 2, 'acct4': 3}
if ACCOUNT not in _ACCOUNT_TO_WORKER:
    raise ValueError(f"ACCOUNT must be one of {list(_ACCOUNT_TO_WORKER)}, got {ACCOUNT!r}")
WORKER_ID = _ACCOUNT_TO_WORKER[ACCOUNT]
if WORKER_ID >= NUM_WORKERS:
    raise ValueError(f"{ACCOUNT} maps to worker {WORKER_ID}, outside NUM_WORKERS={NUM_WORKERS}. "
                     "For a one-notebook run use acct1; for four use acct1..acct4.")

sess = tl.Session(account=ACCOUNT, worker_id=WORKER_ID, num_workers=NUM_WORKERS,
                  stage='ens',
                  hf_repo='Shanmuk4622/tyre-wear-study',
                  enable_hf=True,
                  session_limit_h=8.5,      # push + pause before Kaggle kills us
                  push_interval_min=30)     # background commit cycle


In [ ]:
# === Find the dataset ======================================================
# One Kaggle dataset holds the whole package:
#     <slug>/FINAL/{images,splits,manifests}
#     <slug>/annotations/{clean,propagated}
# Kaggle sometimes wraps uploads in one more directory, so both are searched for.
DATA_ROOT = sess.prepare_data()
ANN_ROOT  = tl.find_annotations_root(DATA_ROOT)
print("annotations:", ANN_ROOT if ANN_ROOT else "NOT FOUND (only needed from NB08 onward)")


In [ ]:
import pandas as pd, numpy as np, torch
from pathlib import Path
from huggingface_hub import hf_hub_download

pull_root = Path(sess.stage_dir) / "ensemble_pull"
try:
    sp = hf_hub_download(tl.HF_REPO_DEFAULT, "tables/stage_b_selection.csv",
                         repo_type="dataset", token=None, local_dir=str(pull_root))
    SEL = pd.read_csv(sp)
    assert ("selection_revision" in SEL and
            SEL.selection_revision.eq("2026-08-30-r3").all())
    flag = (SEL.selected_top3 if SEL.selected_top3.dtype == bool else
            SEL.selected_top3.astype(str).str.lower().eq("true"))
    TOP3 = list(SEL.loc[flag, "arch"])
except Exception as e:
    raise RuntimeError("Run NB07 first; no locked top-three selection exists.") from e
assert len(TOP3) == 3
print("ensemble architectures:", TOP3)


## 2 — Pull the 27 prediction files; reconstruct any derivable gap

In [ ]:
from PIL import Image
import gc

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_or_rebuild_predictions(rid):
    """A best checkpoint is primary evidence; predictions are reproducible.

    The public audit found one historical run whose final metrics and best
    checkpoint exist but whose predictions parquet did not land. Reconstruct
    that derived file if any selected run has the same gap, publish it, and
    continue instead of making the ensemble depend on a missing upload.
    """
    rel = f"runs/{rid}/per_sample/predictions.parquet"
    try:
        return Path(hf_hub_download(tl.HF_REPO_DEFAULT, rel, repo_type="dataset",
                                    token=None, local_dir=str(pull_root)))
    except Exception as first_error:
        print("REBUILD missing predictions:", rid, type(first_error).__name__)

    ck_rel = f"runs/{rid}/checkpoints/ckpt_best.pt"
    ck = Path(hf_hub_download(tl.HF_REPO_DEFAULT, ck_rel, repo_type="dataset",
                              token=None, local_dir=str(pull_root)))
    st = torch.load(ck, map_location="cpu", weights_only=False); cfg = st["config"]
    model = tl.build_model(cfg["arch"], 3, pretrained=False, head=cfg["head_type"],
                           img_size=cfg["input_resolution"])
    model.load_state_dict(st["model"]); model = model.to(dev).eval()
    _, va = tl.load_split(DATA_ROOT, int(cfg["fold"])); va = va.sort_values("image_id")
    tf = tl.build_transforms(cfg["input_resolution"], False, cfg.get("preprocessing", "raw"))
    rows = []
    for r in va.itertuples():
        x = tf(Image.open(DATA_ROOT/r.relative_path).convert("RGB")).unsqueeze(0).to(dev)
        with torch.no_grad(), tl._autocast(dev): logits = model(x)
        pr = (tl.CoralHead.probs(logits.float()) if cfg["head_type"] == "coral"
              else logits.float().softmax(1))[0].cpu().numpy()
        rows.append({"image_id": r.image_id, "session_group": r.session_group,
                     "true": tl.C2I[r.proxy_label], "pred": int(pr.argmax()),
                     "prob_low": pr[0], "prob_mid": pr[1], "prob_high": pr[2]})
    out = Path(sess.stage_dir)/"recovered_predictions"/rid/"predictions.parquet"
    out.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_parquet(out, index=False)
    sess.uploader.enqueue(out, rel, force=True)
    sess.push_now(f"reconstructed predictions {rid}")
    del model; torch.cuda.empty_cache(); gc.collect()
    try: ck.unlink()
    except Exception: pass
    return out

frames = []
selected_ids = [f"a-{a}-base-f{f}-s{s}" for a in TOP3 for f in (0,1,2) for s in (1,2,3)]
for rid in selected_ids:
    p = get_or_rebuild_predictions(rid)
    d = pd.read_parquet(p); d["run_id"] = rid
    d["arch"] = rid.split("-")[1]; d["fold"] = int(rid.split("-f")[1].split("-")[0])
    d["seed"] = int(rid.rsplit("-s",1)[1]); frames.append(d)
P = pd.concat(frames, ignore_index=True)
PROB = ["prob_low", "prob_mid", "prob_high"]
print(f"{len(P)} rows from {P.run_id.nunique()} runs")


## 3 — Single, seed-ensemble, and architecture-ensemble metrics

In [ ]:
def score_groups(df, group, label):
    rows = []
    for keys, g in df.groupby(group):
        avg = g.groupby("image_id")[PROB].mean()
        truth = g.groupby("image_id")["true"].first().loc[avg.index].astype(int)
        met, _ = tl.classification_report_dict(truth.values, avg.values.argmax(1), avg.values, "")
        rows.append({"kind": label, "group": str(keys), "n": len(avg),
                     **{k: met[k] for k in ("f1_macro","qwk","ece","nll","brier")}})
    return pd.DataFrame(rows)

M = pd.concat([score_groups(P, ["arch","fold","seed"], "single"),
               score_groups(P, ["arch","fold"], "seed_ensemble"),
               score_groups(P, ["fold"], "architecture_ensemble")], ignore_index=True)
print(M.groupby("kind")[["f1_macro","qwk","ece","nll"]].mean().round(4).to_string())


## 4 — Horizontal-flip TTA on one fixed checkpoint per architecture

In [ ]:
def tta_one(arch):
    rid = f"a-{arch}-base-f1-s1"; rel = f"runs/{rid}/checkpoints/ckpt_best.pt"
    ck = Path(hf_hub_download(tl.HF_REPO_DEFAULT, rel, repo_type="dataset",
                              token=None, local_dir=str(pull_root)))
    st = torch.load(ck, map_location="cpu", weights_only=False); cfg = st["config"]
    model = tl.build_model(arch, 3, pretrained=False, head=cfg["head_type"],
                           img_size=cfg["input_resolution"])
    model.load_state_dict(st["model"]); model = model.to(dev).eval()
    _, va = tl.load_split(DATA_ROOT, 1); va = va.sort_values("image_id")
    tf = tl.build_transforms(cfg["input_resolution"], False, cfg.get("preprocessing", "raw"))
    raw, tta, y = [], [], []
    for r in va.itertuples():
        img = Image.open(DATA_ROOT/r.relative_path).convert("RGB")
        x0 = tf(img).unsqueeze(0).to(dev); x1 = torch.flip(x0, dims=[3])
        with torch.no_grad(), tl._autocast(dev): l0, l1 = model(x0), model(x1)
        p0 = tl.CoralHead.probs(l0.float()) if cfg["head_type"]=="coral" else l0.float().softmax(1)
        p1 = tl.CoralHead.probs(l1.float()) if cfg["head_type"]=="coral" else l1.float().softmax(1)
        raw.append(p0[0].cpu().numpy()); tta.append(((p0+p1)/2)[0].cpu().numpy())
        y.append(tl.C2I[r.proxy_label])
    out=[]
    for kind, pr in (("single_view",np.asarray(raw)),("hflip_tta",np.asarray(tta))):
        met,_=tl.classification_report_dict(np.asarray(y),pr.argmax(1),pr,"")
        out.append({"arch":arch,"kind":kind,"n":len(y),
                    **{k:met[k] for k in ("f1_macro","qwk","ece","nll")}})
    del model; torch.cuda.empty_cache(); gc.collect()
    try: ck.unlink()
    except Exception: pass
    return out

TTA = pd.DataFrame(sum((tta_one(a) for a in TOP3), []))
print(TTA.round(4).to_string(index=False))


## 5 — Disjoint temperature/conformal/test splits

In [ ]:
def stratified_three_way(y, seed=0):
    rng=np.random.default_rng(seed); parts=[[],[],[]]
    for cls in sorted(np.unique(y)):
        idx=np.where(y==cls)[0]; rng.shuffle(idx)
        for i,v in enumerate(idx): parts[i%3].append(int(v))
    return [np.array(sorted(x),int) for x in parts]

def fit_temperature(probs, y):
    lp=torch.tensor(np.log(np.clip(probs,1e-8,1)),dtype=torch.float32)
    yy=torch.tensor(y,dtype=torch.long); log_t=torch.zeros(1,requires_grad=True)
    opt=torch.optim.LBFGS([log_t],lr=.2,max_iter=80,line_search_fn="strong_wolfe")
    def closure():
        opt.zero_grad(); loss=torch.nn.functional.cross_entropy(lp/log_t.exp(),yy); loss.backward(); return loss
    opt.step(closure); return float(log_t.exp().clamp(.05,20))

def apply_temperature(probs,T):
    z=np.log(np.clip(probs,1e-8,1))/T; z-=z.max(1,keepdims=True)
    e=np.exp(z); return e/e.sum(1,keepdims=True)

def conformal_sets(cal_probs, cal_y, test_probs, alpha=.10):
    scores=1-cal_probs[np.arange(len(cal_y)),cal_y]
    q=np.quantile(scores,min(1,np.ceil((len(scores)+1)*(1-alpha))/len(scores)),method="higher")
    return (1-test_probs)<=q,float(q)

rows_cal=[]; rows_conf=[]
for fold,g in P.groupby("fold"):
    avg=g.groupby("image_id")[PROB].mean(); y=g.groupby("image_id")["true"].first().loc[avg.index].astype(int).values
    probs=avg.values; temp_i,conf_i,test_i=stratified_three_way(y,seed=100+int(fold))
    T=fit_temperature(probs[temp_i],y[temp_i]); pc=apply_temperature(probs,T)
    for kind,pr in (("raw",probs[test_i]),("temperature",pc[test_i])):
        met,_=tl.classification_report_dict(y[test_i],pr.argmax(1),pr,"")
        rows_cal.append({"fold":fold,"kind":kind,"temperature":T,"n":len(test_i),
                         **{k:met[k] for k in ("f1_macro","ece","nll","brier")}})
    sets,q=conformal_sets(pc[conf_i],y[conf_i],pc[test_i])
    rows_conf.append({"fold":fold,"n_cal":len(conf_i),"n_test":len(test_i),"q":q,
                      "coverage_90":float(sets[np.arange(len(test_i)),y[test_i]].mean()),
                      "mean_set_size":float(sets.sum(1).mean()),
                      "abstain_rate":float((sets.sum(1)>1).mean())})
CAL=pd.DataFrame(rows_cal); CONF=pd.DataFrame(rows_conf)
print("temperature scaling\n",CAL.round(4).to_string(index=False))
print("\nconformal sets\n",CONF.round(4).to_string(index=False))


## 6 — Save, push, and finish

In [ ]:
out=Path(sess.stage_dir)/"tables"; out.mkdir(parents=True,exist_ok=True)
for name,df in (("ensemble_metrics.csv",M),("tta.csv",TTA),
                ("calibration.csv",CAL),("conformal.csv",CONF)):
    df.to_csv(out/name,index=False); sess.uploader.enqueue(out/name,f"tables/{name}",force=True)
sess.push_now("ensemble and uncertainty tables complete"); sess.finish()
